# FX Decision Recommendation System - Unified Master Notebook
**Course**: 23CSE452 – Business Analytics  
**Team Members**:
- Srividya Manikandan – FX Data Analysis & Preprocessing
- Aadhithya Bharathi – Business Exposure Modelling
- Kanishkhan – Risk Scoring Engine
- Adwaitha – Forecasting & Scenario Analysis
- Adarsh – Decision Recommendation & Integration (Lead)

## Project Overview
This notebook integrates all phases of our FX Decision Recommendation System. It covers raw data processing from RBI/FBIL, business exposure analysis for different sectors, risk scoring based on market volatility, and exchange rate forecasting using Prophet. The final output is an actionable recommendation for Indian businesses (Importers, Exporters, IT Firms).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error
from datetime import datetime

# File Paths (Robust Detection)
import os
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..")) if "adarsh_part" in os.getcwd() else os.getcwd()
RAW_RBI = os.path.join(BASE_DIR, 'data', 'raw', 'RBI_BankWise(2k16-26).xlsx')
RAW_FBIL = os.path.join(BASE_DIR, 'data', 'raw', 'Reference_Rates.xlsx')
PROCESSED_PATH = os.path.join(BASE_DIR, 'data', 'processed', 'cleaned_fx_data.csv')

print(f"Project Base: {BASE_DIR}")
print("All libraries loaded and environment ready.")

## Section 1: Data Integration & Preprocessing (Srividya's Part)
This section merges data from the RBI and FBIL repositories to create a continuous 10-year dataset of exchange rates.

In [ ]:
def run_preprocessing():
    # Try to load existing processed data first
    if os.path.exists(PROCESSED_PATH):
        print(f"✓ Found existing processed data at {PROCESSED_PATH}. Loading...")
        df = pd.read_csv(PROCESSED_PATH)
        # Check if first column is date and set as index
        if 'Unnamed: 0' in df.columns:
            df = df.rename(columns={'Unnamed: 0': 'Date'})
        if 'Date' in df.columns:
            df['Date'] = pd.to_datetime(df['Date'])
            df = df.set_index('Date')
        return df

    if not os.path.exists(RAW_RBI) or not os.path.exists(RAW_FBIL):
        print(f"Error: Raw data files not found at {RAW_RBI} or {RAW_FBIL}")
        return None

    # 1. Load and Clean RBI Data
    df_rbi = pd.read_excel(RAW_RBI)
    df_rbi['Date'] = pd.to_datetime(df_rbi['Date'], dayfirst=True)
    df_rbi = df_rbi.rename(columns={'USD (INR / 1 USD)': 'USD', 'GBP (INR / 1 GBP)': 'GBP', 'EUR (INR / 1 EUR)': 'EUR', 'JPY (INR / 100 JPY)': 'JPY'})
    df_rbi = df_rbi[['Date', 'USD', 'GBP', 'EUR', 'JPY']].set_index('Date')

    # 2. Load and Standardize FBIL Data
    df_fbil = pd.read_excel(RAW_FBIL, skiprows=2)
    df_fbil['Date'] = pd.to_datetime(df_fbil['Date'], dayfirst=True)
    mapping = {'INR / 1 USD': 'USD', 'INR/1 USD': 'USD', 'INR / 1 GBP': 'GBP', 'INR / 1 EUR': 'EUR', 'INR / 100 JPY': 'JPY'}
    df_fbil['Currency'] = df_fbil['Currency Pairs'].map(mapping)
    df_fbil = df_fbil.dropna(subset=['Currency'])
    df_fbil_pivot = df_fbil.pivot_table(index='Date', columns='Currency', values='Rate')

    # 3. Merge and fill gaps
    df_combined = df_rbi.combine_first(df_fbil_pivot).sort_index()
    full_range = pd.date_range(start=df_combined.index.min(), end=df_combined.index.max(), freq='D')
    df_final = df_combined.reindex(full_range).ffill()

    # 4. Calculate Returns and Volatility
    for cur in ['USD', 'GBP', 'EUR', 'JPY']:
        df_final[f'{cur}_Return'] = df_final[cur].pct_change()
        df_final[f'{cur}_Volatility'] = df_final[f'{cur}_Return'].rolling(window=30).std()

    os.makedirs(os.path.dirname(PROCESSED_PATH), exist_ok=True)
    df_final.to_csv(PROCESSED_PATH)
    return df_final

df_master = run_preprocessing()
if df_master is not None:
    print("✓ Master dataset ready. Rows:", len(df_master))
    display(df_master.tail())
else:
    print("❌ Preprocessing Failed. Check data paths.")

## Section 2: Business Exposure Analytics (Aadhithya's Part)
Calculating the financial impact of FX movements on Importers (risk in appreciation), Exporters (risk in depreciation), and IT Firms (margin risk).

In [ ]:
def calculate_business_impact(base_usd, current_rate, change_percent):
    changed_rate = current_rate * (1 + change_percent/100)
    diff = changed_rate - current_rate
    
    results = {
        "Business": ["Importer", "Exporter", "IT Services"],
        "USD_Exposure": [base_usd, base_usd, base_usd],
        "Impact_INR": [
            base_usd * diff,       # Cost increases for importers
            base_usd * -diff,      # Revenue decreases for exporters if INR strengthens
            (base_usd * diff)      # Revenue change for IT firm
        ]
    }
    return pd.DataFrame(results)

if df_master is not None:
    latest_usd = df_master['USD'].iloc[-1]
    exposure_df = calculate_business_impact(100000, latest_usd, 1.0)
    print(f"\nExposure Impact for $100k move at rate {latest_usd:.2f}:")
    display(exposure_df)

## Section 3: Risk Scoring & Recommendation Engine (Kanishkhan's Part)
This section implements the automated risk scoring logic (60% Market Volatility, 40% Business Exposure) and flags critical risk periods.

In [ ]:
def calculate_risk_scores(df):
    risk_df = df.copy()
    
    # Standardize exposure (simulate 100k-500k range behavior)
    exp_sim = 250000 
    
    # 1. Volatility Score (0-100)
    v_min, v_max = risk_df['USD_Volatility'].min(), risk_df['USD_Volatility'].max()
    latest_v = risk_df['USD_Volatility'].iloc[-1]
    vol_score = ((latest_v - v_min) / (v_max - v_min)) * 100
    
    # 2. Exposure Score (0-100 normalized range)
    exp_score = ((exp_sim - 50000) / (500000 - 50000)) * 100
    
    # 3. Final Weighted Score
    final_score = (0.6 * vol_score) + (0.4 * exp_score)
    
    # Risk Categorization
    risk_lvl = "Low" if final_score < 40 else "Medium" if final_score <= 70 else "High"
    
    # Alert Logic
    alert = "🚨 ACTION REQUIRED" if risk_lvl == "High" else "Monitor"
    
    return {"Score": final_score, "Level": risk_lvl, "Alert": alert}

if df_master is not None:
    current_risk = calculate_risk_scores(df_master)
    print(f"\n--- LIVE RISK STATUS ---")
    print(f"Risk Score: {current_risk['Score']:.2f}")
    print(f"Risk Level: {current_risk['Level']}")
    print(f"Status: {current_risk['Alert']}")

## Section 4: FX Rate Forecasting (Adwaitha's Part)
Using Prophet to predict exchange rate movement for the next 7 days to assist in decision timing.

In [ ]:
if df_master is not None:
    # Prepare Prophet Data
    pdf = df_master.reset_index()[['Date', 'USD']]
    pdf.columns = ['ds', 'y']

    # Fit Model
    m = Prophet(daily_seasonality=False, weekly_seasonality=True, yearly_seasonality=True)
    m.fit(pdf)

    # Forecast 7 Days
    future = m.make_future_dataframe(periods=7)
    forecast = m.predict(future)

    # Plot
    fig = m.plot(forecast)
    plt.title("USD-INR exchange Rate Forecast")
    plt.show()

    latest_pred = forecast['yhat'].iloc[-1]
    print(f"Predicted USD Rate in 7 days: ₹{latest_pred:.4f}")

## Section 5: Consolidated Decision Dashboard
Final output for management review.

In [ ]:
if df_master is not None:
    print("="*50)
    print("     FX DECISION RECOMMENDATION ENGINE")
    print("="*50)
    print(f"Current Spot Rate:  ₹{latest_usd:.4f}")
    print(f"7-Day Forecast:     ₹{latest_pred:.4f}")
    print(f"Market Volatility:  {df_master['USD_Volatility'].iloc[-1]:.6f}")
    print(f"Risk Assessment:    {current_risk['Level'].upper()}")
    print("-"*50)

    trend = "UP" if latest_pred > latest_usd else "DOWN"

    if current_risk['Level'] == "High":
        rec = "HEDGE IMMEDIATELY - Forward Contract Recommended"
    elif trend == "UP" and current_risk['Level'] != "Low":
        rec = "CONVERT 50% - Partial Hedge due to rising trend"
    else:
        rec = "WAIT / SPOT CONVERSION - No immediate risk"
        
    print(f"FINAL RECOMMENDATION: {rec}")
    print("="*50)